# Часть 0. Загрузка и предобработка данных

## Шаг 1. Импорты

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn

## Шаг 2. Скачивание датасета

Перейдите по ссылке из личного кабинета на Google Таблицу со списком студентов. Найдите своё ФИО в списке и запомните соответствующий порядковый номер. Заполните его в ячейке ниже и выполните ячейку. Если вы не можете найти себя в списке, обратитесь к своему преподавателю.

In [2]:
### BEGIN YOUR CODE

Student_ID = 24

### END YOUR CODE

Теперь выполните следующую ячейку. Она вычислит номер задания и выведет его.

In [3]:
datasets = [
    (
        'Combined Cycle Power Plant',
        'https://archive.ics.uci.edu/dataset/294/combined+cycle+power+plant',
        'PE',
        85, 15
    ),
    (
        'Beijing PM2.5',
        'https://archive.ics.uci.edu/dataset/381/beijing+pm2+5+data',
        'pm2.5',
        80, 20
    ),
    (
        'Airfoil Self-Noise',
        'https://archive.ics.uci.edu/dataset/291/airfoil+self+noise',
        'scaled-sound-pressure',
        90, 10
    ),
    (
        'Energy Efficiency',
        'https://archive.ics.uci.edu/dataset/242/energy+efficiency',
        'Y1, Y2 (здесь две целевые переменные)',
        90, 10
    ),
    (
        'Bike Sharing',
        'https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset',
        'cnt',
        90, 10
    ),
    (
        'Forest Fires',
        'https://archive.ics.uci.edu/dataset/162/forest+fires',
        'area',
        90, 10
    ),
    (
        'Concrete Compressive Strength',
        'https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength',
        'Concrete compressive strength',
        90, 10
    )
]

dataset_id = None if Student_ID is None else (Student_ID - 1) % len(datasets)
if dataset_id is None:
    print("ОШИБКА! Не указан порядковый номер студента в списке группы.")
else:
    print(f"Информация о датасете '{datasets[dataset_id][0]}' доступна по следующей ссылке: {datasets[dataset_id][1]}")
    print(f"Целевая переменная: {datasets[dataset_id][2]}")
    print(f"Разделение выборки на train+val/test: {datasets[dataset_id][3]}%/{datasets[dataset_id][4]}%")

Информация о датасете 'Airfoil Self-Noise' доступна по следующей ссылке: https://archive.ics.uci.edu/dataset/291/airfoil+self+noise
Целевая переменная: scaled-sound-pressure
Разделение выборки на train+val/test: 90%/10%


## Шаг 3. Загрузка и обработка датасета

Скачайте датасет по ссылке выше и считайте его в датафрейм `df`.

Большинство датасетов можно загрузить через `pd.read_csv(...)`. Если файл архивирован — распакуйте его вручную или воспользуйтесь `zipfile` / `requests`.

In [5]:
### BEGIN YOUR CODE


df = pd.read_csv(
    r"/content/airfoil_self_noise.dat",
    sep="\t",
    header=None
)


### END YOUR CODE

Обработайте датасет. В обработке вы имеете почти полную свободу — важно побить бейзлайн.

**Что НЕОБХОДИМО сделать** (иначе тесты не пройдут):
- Обработать пропуски (удалить строки / заполнить средним / медианой — на ваш выбор)
- Закодировать категориальные признаки (например, `pd.get_dummies` или `LabelEncoder`)

**Что ПОЛЕЗНО сделать** (помогает побить бейзлайн):
- Удалить дубликаты
- Обработать выбросы
- Удалить признаки с низкой информативностью (посмотрите на корреляции)
- Добавить новые признаки (feature engineering)
- Стандартизировать числовые признаки

> **Подсказка:** [sklearn.preprocessing](https://scikit-learn.org/stable/api/sklearn.preprocessing.html) содержит `StandardScaler`, `MinMaxScaler`, `LabelEncoder`, `OneHotEncoder` и другие полезные инструменты.

> ⚠️ **Важно про стандартизацию целевой переменной:** если вы стандартизируете `y`, перед сохранением результатов в `y_pred_*` нужно вернуть предсказания к исходному масштабу. Иначе MAPE будет считаться неверно и тесты провалятся. Сохраните параметры стандартизации (`mean`, `std`) для обратного преобразования.

**ВАЖНО:** обработанный по обязательным пунктам датафрейм запишите в `df_base`, дополнительно улучшенный — в `df_processed`. Не перезаписывайте эти переменные и `df`.

In [6]:
### BEGIN YOUR CODE


df_base = df.copy()

df_base.columns = ["frequency", "angle", "chord", "velocity", "thickness", "scaled-sound-pressure"]

df_base = df_base.dropna()
df_base = pd.get_dummies(df_base)


### END YOUR CODE

In [7]:
### BEGIN YOUR CODE


df_processed = df_base.copy()

df_processed = df_processed.drop_duplicates()
df_processed["frequency_squared"] = df_processed["frequency"] ** 2


### END YOUR CODE

# Часть 1. Линейная регрессия (scikit-learn)

## Что такое линейная регрессия?

Линейная регрессия — это модель, которая предсказывает числовое значение `y` как **линейную комбинацию признаков** `x`:

$$y = \mathbf{x} \boldsymbol{\theta} + \theta_0$$

где $\boldsymbol{\theta}$ — вектор весов (коэффициентов), $\theta_0$ — свободный член (смещение, bias/intercept).

**Как модель обучается?** Она подбирает $\boldsymbol{\theta}$ и $\theta_0$ так, чтобы минимизировать **полную функцию потерь** (функционал качества, эмпирический риск). Чаще всего используют **MSE (Mean Squared Error)**:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Чем меньше MSE — тем ближе предсказания к реальным значениям.

**Как оценивать качество?** В этой работе используется **MAPE (Mean Absolute Percentage Error)**:

$$\text{MAPE} = \frac{1}{n} \sum_{i=1}^{n} \left| \frac{y_i - \hat{y}_i}{y_i} \right|$$

MAPE удобна тем, что выражается в процентах: MAPE = 0.05 означает среднюю ошибку 5%. Она позволяет сравнивать качество моделей на разных датасетах с разными масштабами целевой переменной.

## Шаг 1. Разделение на train / val / test

**Зачем три выборки?**

| Выборка | Назначение |
|---------|------------|
| **Train** | Обучение модели (подбор весов) |
| **Val (validation)** | Подбор гиперпараметров (например, скорость обучения, количество эпох) без «подглядывания» в тест |
| **Test** | Финальная оценка качества — трогается **только один раз** в самом конце |

Если настраивать гиперпараметры сразу по test, то модель фактически «видит» тестовые данные и реальная ошибка на новых данных окажется хуже.

> **Размер test-выборки** указан в вашем варианте задания (выше, рядом со ссылкой на датасет). Используйте `train_test_split` из sklearn.

In [8]:
### BEGIN YOUR CODE

# Разделите df_base и df_processed на матрицу признаков X и вектор целевой переменной y
X_base = df_base.drop("scaled-sound-pressure", axis=1)
y_base = df_base["scaled-sound-pressure"]

X_processed = df_processed.drop("scaled-sound-pressure", axis=1)
y_processed = df_processed["scaled-sound-pressure"]


### END YOUR CODE

In [9]:
from sklearn.model_selection import train_test_split

### BEGIN YOUR CODE

# Для base: только train и test (val не нужен, гиперпараметров нет)

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(X_base, y_base, test_size=0.1, random_state=42)


### END YOUR CODE

In [10]:
### BEGIN YOUR CODE

# Для processed: сначала отделить test, затем из оставшегося выделить val

X_train_processed, X_test_processed, y_train_processed, y_test_processed = train_test_split(
    X_processed, y_processed, test_size=0.1, random_state=42
)

X_train_processed, X_val_processed, y_train_processed, y_val_processed = train_test_split(
    X_train_processed, y_train_processed, test_size=0.2, random_state=42
)

### END YOUR CODE

In [11]:
# Промежуточная проверка — убедитесь, что размеры выглядят разумно
print("Base:")
print(f"  X_train_base:    {X_train_base.shape}")
print(f"  X_test_base:     {X_test_base.shape}")
print()
print("Processed:")
print(f"  X_train_processed: {X_train_processed.shape}")
print(f"  X_val_processed:   {X_val_processed.shape}")
print(f"  X_test_processed:  {X_test_processed.shape}")

Base:
  X_train_base:    (1352, 5)
  X_test_base:     (151, 5)

Processed:
  X_train_processed: (1081, 6)
  X_val_processed:   (271, 6)
  X_test_processed:  (151, 6)


## Шаг 2. Обучение линейной регрессии

Ниже приведён **бейзлайн** — простейшая линейная регрессия без какой-либо дополнительной обработки данных. Ваша задача — **побить его**, то есть получить `MAPE(y_pred_processed) < MAPE(y_pred_base)`.

Для этого используйте `df_processed` и экспериментируйте с:
- предобработкой данных (стандартизация, удаление выбросов, feature engineering)
- подбором гиперпараметров на `val`-выборке

> **Подсказка:** гиперпараметры подбираются на val, финальная оценка считается на test. Не оптимизируйте напрямую по test!

In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error

In [13]:
def baseline():
    """Бейзлайн: LinearRegression без предобработки, обученный на base-данных."""
    lin_reg = LinearRegression()
    lin_reg.fit(X_train_base, y_train_base)
    return lin_reg.predict(X_test_base)

In [14]:
y_pred_base = baseline()
print(f"Baseline MAPE: {mean_absolute_percentage_error(y_test_base, y_pred_base):.4f}")

Baseline MAPE: 0.0298


Обучите улучшенную модель и запишите предсказания на тестовой выборке в `y_pred_processed`.

In [15]:
### BEGIN YOUR CODE


model = LinearRegression()

model.fit(X_train_processed, y_train_processed)

y_pred_processed = model.predict(X_test_processed)


### END YOUR CODE

print(f"Processed MAPE: {mean_absolute_percentage_error(y_test_processed, y_pred_processed):.4f}")
print(f"Baseline MAPE:  {mean_absolute_percentage_error(y_test_base, y_pred_base):.4f}")

Processed MAPE: 0.0290
Baseline MAPE:  0.0298


# Часть 2. Градиентный спуск (numpy)

## Что такое градиентный спуск?

Представьте, что полная функция потерь — это холмистый ландшафт, а мы стоим на нём и хотим спуститься в самую низкую точку (минимум). Градиент указывает направление наибольшего **роста** функции, поэтому мы идём в противоположном направлении:

$$\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \eta \cdot \nabla_{\boldsymbol{\theta}} Q$$

где $\eta$ — **learning rate** (размер шага), $\nabla_{\boldsymbol{\theta}} Q$ — градиент полной функции потерь по весам.

**Стохастический** мини-батчевый градиентный спуск (SGD) отличается тем, что градиент вычисляется не на всей выборке, а на случайном **мини-батче** — случайной подвыборке небольшого размера. Это быстрее и часто сходится лучше.

---

### Градиенты MSE для линейной модели

Формулы градиентов $\dfrac{\partial Q}{\partial \boldsymbol{\theta}}$ и $\dfrac{\partial Q}{\partial \theta_0}$ для модели $\hat{y} = X\boldsymbol{\theta} + \theta_0$ и полной функции потерь $Q = \frac{1}{n}\|\hat{y} - y\|^2$ выводятся в теоретической части (вопрос Q1).

Именно их нужно реализовать в методе `step()` класса `CustomSGD`.

> **Интерфейс:** `CustomSGD.__init__(params, lr)` принимает список параметров модели `[W, b]` и хранит их как атрибут, где `W` — "веса" модели, т.е. параметры $\boldsymbol{\theta}$; а `b` — смещение $\theta_0$. Метод `step(x_batch, y_batch)` вычисляет градиенты и обновляет параметры **in-place** (через `params[0] -= ...`).

## Шаг 1. Реализация CustomSGD

In [ ]:
### BEGIN YOUR CODE
class CustomSGD:
    def __init__(self, params, lr):
        """
        params: список параметров модели, например [W, b]
                Изменения в params автоматически отражаются в модели,
                поэтому достаточно хранить ссылку на этот список.
        lr: learning rate (скорость обучения), положительное число
        """
        pass

    def step(self, x_batch, y_batch):
        """
        Один шаг SGD: вычислить градиенты MSE на батче и обновить params in-place.

        x_batch: numpy array размера (batch_size, n_features)
        y_batch: numpy array размера (batch_size,)
        """
        pass

### END YOUR CODE

# Часть 3. Линейная регрессия (numpy)

## Шаг 1. Реализация CustomLinearRegression

Реализуйте класс линейной регрессии на numpy. Класс должен поддерживать два режима обучения, задаваемых параметром `solver`:

- `solver='SGD'` — обучение через `CustomSGD` (стохастический градиентный спуск)
- `solver='MNK'` — обучение методом наименьших квадратов (аналитическое решение)

**Метод наименьших квадратов (МНК)** даёт точное решение задачи минимизации MSE без итераций:

$$\boldsymbol{\theta} = (X^\top X)^{-1} X^\top y$$

Для его реализации удобно расширить матрицу признаков столбцом единиц (`np.hstack([X, np.ones(...)])`), тогда последний элемент вектора весов будет соответствовать свободному члену $b$.

При реализации через numpy используйте `np.linalg.solve(A, b)` (вместо явного обращения матрицы через `np.linalg.inv`) — это устойчивее численно.

> **Бонусный параметр** (для бонусного задания *): `regularization` задаёт тип регуляризации (`'ridge'` или `'lasso'`), `tau` — её коэффициент.

In [ ]:
class CustomLinearRegression:
### BEGIN YOUR CODE
    """
    Линейная регрессия y = Xw + b, реализованная на numpy.

    Параметры:
        solver         : 'SGD' (градиентный спуск) или 'MNK' (метод наименьших квадратов)
        regularization : None, 'ridge' (L2) или 'lasso' (L1) — только для solver='SGD'
        tau            : коэффициент регуляризации (используется при regularization != None)
    """
    def __init__(self, solver='SGD', regularization=None, tau=0.01):
        pass

    def fit(self, X, y):
        """
        Обучить модель.
        X: numpy array (n_samples, n_features)
        y: numpy array (n_samples,) или (n_samples, n_targets)
        """
        pass

    def predict(self, X):
        """
        Вернуть предсказания для X.
        X: numpy array (n_samples, n_features)
        """
        pass

### END YOUR CODE

## Шаг 2. Обучение CustomLinearRegression с помощью SGD

Обучите модель с помощью `CustomSGD` и запишите предсказания на тестовой выборке в `y_pred_custom`.

**Что нужно сделать:**
1. Инициализировать параметры модели `W` и `b`
2. Реализовать мини-батчевый обучающий цикл (эпохи → батчи → шаг оптимизатора) — он ляжет в основу `fit()` в `CustomLinearRegression`, но здесь дополнительно отслеживается val loss
3. На каждой эпохе сохранять средний MSE на train и val выборках
4. После обучения построить графики функции потерь на train и val выборках
5. Вычислить MAPE на тестовой выборке и сравнить с sklearn-бейзлайном

Ниже приведён шаблон обучающего цикла — заполните недостающие части.

In [ ]:
# Шаблон обучающего цикла — заполните недостающие части

n_epochs = 100
batch_size = 32
learning_rate = 0.01

### BEGIN YOUR CODE

# Инициализируйте параметры модели (веса и bias)
# Подсказка: начните с нулей или случайных маленьких значений
W = ...  # размер (n_features, 1) или (n_features,)
b = ...  # скаляр или массив (1,)
params = [W, b]

optimizer = CustomSGD(params, lr=learning_rate)

train_losses = []
val_losses = []

for epoch in range(n_epochs):
    # Реализуйте мини-батчевый SGD:
    # 1. Перемешайте индексы тренировочных данных
    # 2. Разбейте на батчи размером batch_size
    # 3. Для каждого батча сделайте шаг оптимизатора и накопите loss
    epoch_loss = ...
    train_losses.append(epoch_loss)

    # Вычислите val loss на X_val_processed, y_val_processed
    val_losses.append(...)

### END YOUR CODE

# График функции потерь
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train loss')
plt.plot(val_losses, label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Кривая обучения')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
### BEGIN YOUR CODE

y_pred_custom = ...

### END YOUR CODE

print(f"Custom SGD MAPE: {mean_absolute_percentage_error(y_test_processed, y_pred_custom):.4f}")
print(f"Baseline MAPE:   {mean_absolute_percentage_error(y_test_base, y_pred_base):.4f}")

## Шаг 3. Метод наименьших квадратов (МНК)

Обучите `CustomLinearRegression` с `solver='MNK'` на вашем датасете.

МНК даёт **точное** решение без итераций — это удобно для небольших датасетов. На очень больших данных инвертирование матрицы $(X^\top X)$ может быть медленным или численно нестабильным — в таком случае возьмите случайную долю данных.

Сравните результаты МНК с SGD: они должны быть близкими (теоретически МНК — глобальный минимум MSE).

Запишите предсказания в `y_pred_mnk`.

In [ ]:
### BEGIN YOUR CODE

y_pred_mnk = ...

### END YOUR CODE

print(f"МНК MAPE:      {mean_absolute_percentage_error(y_test_processed, y_pred_mnk):.4f}")
print(f"Custom SGD MAPE: {mean_absolute_percentage_error(y_test_processed, y_pred_custom):.4f}")
print(f"Baseline MAPE:   {mean_absolute_percentage_error(y_test_base, y_pred_base):.4f}")

In [ ]:
# Проверка реализации МНК
sk_model = LinearRegression().fit(X_train_processed, y_train_processed)
y_pred_sk = sk_model.predict(X_test_processed)

assert np.allclose(y_pred_mnk, y_pred_sk, atol=1e-4), \
    "Проверьте реализацию МНК: предсказания расходятся с эталонной реализацией sklearn."
print("✓ МНК реализован корректно")

> Примечание о числовой точности. `np.linalg.solve` устойчивее явного `inv`, но оба метода работают с матрицей $X^\top X$, число обусловленности которой равно **квадрату** числа обусловленности исходной матрицы $X$. При добавлении полиномиальных признаков или признаков с сильно различающимися масштабами это может достигать $10^{16}$ — предела точности `float64`. sklearn внутри использует SVD и не формирует $X^\top X$ явно, поэтому его результаты могут отличаться от `solve` на величину порядка $10^{-3}$–$10^{-2}$. Это не ошибка реализации, а ожидаемое следствие разных численных методов. Если проверка не проходит — убедитесь, что логика верна, и при необходимости увеличьте `atol`.

# Бонус *. Регуляризация (Ridge и LASSO) [+10 баллов]

**Зачем нужна регуляризация?**

Модель может переобучиться: идеально подстроиться под тренировочные данные, но плохо работать на новых. Регуляризация добавляет к функции потерь **штраф за большие веса**, что вынуждает модель оставаться проще.

**Ridge (L2-регуляризация):**

$$Q_{\text{Ridge}} = \text{MSE} + \tau \|\boldsymbol{\theta}\|_2^2 = \text{MSE} + \tau \sum_j \theta_j^2$$

**LASSO (L1-регуляризация):**

$$Q_{\text{LASSO}} = \text{MSE} + \tau \|\boldsymbol{\theta}\|_1 = \text{MSE} + \tau \sum_j |\theta_j|$$

LASSO обнуляет незначимые веса (делает модель разреженной), Ridge лишь уменьшает их.

Дополнительные слагаемые в градиенте выводятся в теоретической части (вопрос Q2). Используйте их при реализации шага SGD с регуляризацией.

**Что нужно сделать:** добавить поддержку регуляризации в `CustomLinearRegression`. Интерфейс:

```python
model_ridge = CustomLinearRegression(solver='SGD', regularization='ridge', tau=0.01)
model_lasso = CustomLinearRegression(solver='SGD', regularization='lasso', tau=0.01)
```

Запишите предсказания в `y_pred_ridge` и `y_pred_lasso`.

In [ ]:
### BEGIN YOUR CODE

y_pred_ridge = ...
y_pred_lasso = ...

### END YOUR CODE

print(f"Ridge MAPE:    {mean_absolute_percentage_error(y_test_processed, y_pred_ridge):.4f}")
print(f"LASSO MAPE:    {mean_absolute_percentage_error(y_test_processed, y_pred_lasso):.4f}")
print(f"Baseline MAPE: {mean_absolute_percentage_error(y_test_base, y_pred_base):.4f}")

# Бонус **. Оптимизатор Adam [+20 баллов]

Adam (Adaptive Moment Estimation) — улучшение SGD, которое адаптирует learning rate для каждого параметра отдельно. Он комбинирует идеи **Momentum** (накопление градиентов) и **RMSProp** (нормировка по скользящей дисперсии):

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t \quad \text{(первый момент)}$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 \quad \text{(второй момент)}$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t} \quad \text{(коррекция смещения)}$$
$$\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \varepsilon}$$

Типичные значения гиперпараметров: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\varepsilon = 10^{-8}$.

Реализуйте `CustomAdam` и обучите `CustomLinearRegression` с его помощью. Запишите предсказания в `y_pred_adam`.

In [ ]:
class CustomAdam:
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def step(self, x_batch, y_batch):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

Обучите `CustomLinearRegression` с помощью `CustomAdam` (аналогично шагу 2 части 3).

In [ ]:
### BEGIN YOUR CODE

y_pred_adam = ...

### END YOUR CODE

print(f"Adam MAPE:     {mean_absolute_percentage_error(y_test_processed, y_pred_adam):.4f}")
print(f"Baseline MAPE: {mean_absolute_percentage_error(y_test_base, y_pred_base):.4f}")